In [ ]:
import os
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline
from peft import get_peft_model, LoraConfig

# Configuration
OUTPUT_DIR = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_Model_Imagea_generation_with_prompt"
CSV_FILENAME = "Pneumonia_50_to_70_male_white.csv"
MODEL_PATH = "runwayml/stable-diffusion-v1-5"
LORA_WEIGHTS_PATH = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_prompt_guided_fine_tuning_stable_diffusion_model/lora_unet_epoch_28.pt"
NUM_IMAGES = 250
PROMPT = "chest X-ray of a 50 to 70 years male white patient showing Pneumonia"
PATIENT_ID_START = 1500
PATIENT_ID_END = 1750

# Function to load LoRA weights
def load_lora_weights(pipe, lora_path):
    lora_config = LoraConfig(
        r=4,
        lora_alpha=16,
        target_modules=["to_q", "to_k", "to_v"],
        lora_dropout=0.1,
        bias="none"
    )
    pipe.unet = get_peft_model(pipe.unet, lora_config)
    pipe.unet.load_state_dict(torch.load(lora_path), strict=False)
    pipe.unet.eval()
    print(f"✅ Loaded LoRA weights from: {lora_path}")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the fine-tuned model
pipe = StableDiffusionPipeline.from_pretrained(MODEL_PATH, torch_dtype=torch.float32).to("cuda")
load_lora_weights(pipe, LORA_WEIGHTS_PATH)

# Initialize list to store metadata
metadata = []

# Define disease columns
diseases = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"
]

# Ensure that the built-in range function is not overwritten
if isinstance(range, tuple):
    del range

for i in range(PATIENT_ID_START, PATIENT_ID_END):
    # Generate image
    image = pipe(PROMPT).images[0]

    # Create a file path in the CheXpert style
    patient_id = f"patient{str(i+1).zfill(5)}"
    study_id = f"study{str(i % 10 + 1)}"
    view = f"view{i % 3 + 1}_frontal.jpg"
    file_path = f"CheXpert-v1.0/train/{patient_id}/{study_id}/{view}"

    # Save image
    save_path = os.path.join(OUTPUT_DIR, file_path)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    image.save(save_path)

    # Extract metadata from the prompt
    age_group = "50-70"
    gender = "Male"
    race = "White"
    disease_presence = {d: 0 for d in diseases}
    disease_presence['Pneumonia'] = 1

    # Append metadata to the list
    metadata.append({
        "Path": file_path,
        "Sex": gender,
        "Age": age_group,
        **disease_presence,
        "PRIMARY_RACE": race
    })

# Save metadata to a CSV file
df = pd.DataFrame(metadata)
df.to_csv(CSV_FILENAME, index=False)
print(f"Generated {PATIENT_ID_END - PATIENT_ID_START} images and saved metadata to {CSV_FILENAME}.")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score, precision_score, recall_score, brier_score_loss

# Step 1: Define transformations for the X-ray images
transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Step 2: Define the CheXpert Dataset for testing
class CheXpertTestDataset(Dataset):
    def __init__(self, dataframe, image_root, transform=None):
        self.dataframe = dataframe
        self.image_root = image_root
        self.transform = transform

        # Disease labels
        self.disease_cols = [
            'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
            'Edema', 'Consolidation', 'Pneumonia',
            'Pneumothorax', 'Pleural Effusion', 'Fracture', 'Support Devices'
        ]

        # Demographic labels
        self.demographic_cols = ['Sex', 'Age', 'PRIMARY_RACE']

        # Combine disease and demographic labels
        self.label_cols = self.disease_cols + self.demographic_cols

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        item = self.dataframe.iloc[idx]
        img_path = os.path.join(self.image_root, item['Path'])

        # Load image
        try:
            image = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, IOError):
            print(f"Warning: Failed to load image {img_path}")
            return None

        if self.transform:
            image = self.transform(image)

        # Extract labels and handle categorical variables
        labels = []
        for col in self.label_cols:
            value = item[col]
            if col in ['Sex', 'Age', 'PRIMARY_RACE']:
                # Convert categorical values to numerical
                value = 1 if str(value).lower() in ['male', '50-70', 'white'] else 0
            labels.append(value)

        label = torch.tensor(labels, dtype=torch.float32)
        return image, label

# Step 3: Load the model checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torchvision.models import resnet50

model = resnet50(pretrained=False)
num_features = model.fc.in_features
model.fc = torch.nn.Linear(num_features, 14)
model = model.to(device)
model.load_state_dict(torch.load("/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/resnet50_Real_vs_real_training_for_14_disease/model_epoch_5.pth", map_location=device))
model.eval()

test_df = pd.read_csv("/home/dawood/lab2_rotaion/No Finding_50_to_70_male_white.csv")
image_root = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/LORA_Model_Imagea_generation_with_prompt"


# Step 5: Create DataLoader for testing
test_dataset = CheXpertTestDataset(test_df, image_root, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Step 6: Evaluation function
def evaluate_model(model, data_loader):
    all_labels = []
    all_outputs = []

    with torch.no_grad():
        for batch in data_loader:
            if batch is None:
                continue
            images, labels = batch
            images = images.to(device)
            outputs = model(images)
            all_labels.append(labels.cpu().numpy())
            all_outputs.append(outputs.cpu().numpy())

    all_labels = np.concatenate(all_labels, axis=0)
    all_outputs = np.concatenate(all_outputs, axis=0)
    all_outputs = 1 / (1 + np.exp(-all_outputs))

    for i, label in enumerate(test_dataset.label_cols):
        y_true = all_labels[:, i]
        y_pred = all_outputs[:, i]
        y_pred_label = (y_pred >= 0.5).astype(int)

        try:
            auc = roc_auc_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred_label, zero_division=0)
            recall = recall_score(y_true, y_pred_label, zero_division=0)
            bce = brier_score_loss(y_true, y_pred)
            print(f"{label}: AUC={auc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, BCE={bce:.4f}")
        except ValueError:
            print(f"{label}: Insufficient data for metrics calculation.")

# Step 7: Run the evaluation
evaluate_model(model, test_loader)